In [ ]:
import pandas as pd
import numpy as np
import os
import pickle
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import LabelEncoder

In [ ]:
# Load Data
df = pd.read_csv('07_employee_attrition/data/hr_attrition_simulated.csv')

In [ ]:
# Preprocessing
X = df.drop('Attrition', axis=1)
y = (df['Attrition'] == 'Yes').astype(int)

In [ ]:
# Categorical Encoding
cat_cols = ['OverTime', 'MaritalStatus']
le_dict = {}
for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    le_dict[col] = le

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
# 1. Interpretable Decision Tree
print("Training Interpretable Decision Tree...")
dt_model = DecisionTreeClassifier(max_depth=4, random_state=42)
dt_model.fit(X_train, y_train)

In [ ]:
# Export Rules
tree_rules = export_text(dt_model, feature_names=list(X.columns))
with open('07_employee_attrition/models/tree_rules.txt', 'w') as f:
    f.write(tree_rules)

In [ ]:
# 2. Predictive Random Forest
print("Training Random Forest...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

In [ ]:
# Evaluation
y_pred_dt = dt_model.predict_proba(X_test)[:, 1]
y_pred_rf = rf_model.predict_proba(X_test)[:, 1]
print(f"Decision Tree AUC: {roc_auc_score(y_test, y_pred_dt):.4f}")
print(f"Random Forest AUC: {roc_auc_score(y_test, y_pred_rf):.4f}")

In [ ]:
# Save Models
os.makedirs('07_employee_attrition/models', exist_ok=True)
with open('07_employee_attrition/models/dt_model.pkl', 'wb') as f:
    pickle.dump(dt_model, f)

In [ ]:
with open('07_employee_attrition/models/rf_model.pkl', 'wb') as f:
    pickle.dump(rf_model, f)

In [ ]:
print("Project 07 Models Successfully Trained.")